In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR


df = pd.read_excel("./COMBINED C-PEP _ ISR.xlsx", sheet_name="combined")
def create_lag_features(df, lags=3):
    df_lag = df.copy()
    for lag in range(1, lags + 1):
        df_lag[f"CPeptide_lag{lag}"] = df_lag.groupby("Sample ID")["C-Peptide"].shift(lag)
    return df_lag.dropna()

df_lagged = create_lag_features(df)

X_rf = df_lagged[['Time (min)', 'C-Peptide', 'CPeptide_lag1', 'CPeptide_lag2', "CPeptide_lag3"]]
y_rf = df_lagged["ISR"]
X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(X_rf, y_rf, test_size=0.2, random_state=42)
X_train_rf

base_models = [
    ("rdf", RandomForestRegressor( n_estimators=100, max_depth=5, random_state=42, max_features="sqrt")),
    ("xgb", XGBRegressor(n_estimators=100, learning_rate=0.5, max_depth=5,)),
]

meta_model = LinearRegression()

stacked_model = StackingRegressor(
    estimators = base_models,
    final_estimator = meta_model,
    passthrough=False
)

prediction_model = Pipeline(steps=[("model", stacked_model)])
prediction_model.fit(X_train_rf, y_train_rf)

prediction = prediction_model.predict(X_test_rf)

print("Random Forest RMSE:",
      np.sqrt(mean_squared_error(y_test_rf, prediction)))


Random Forest RMSE: 0.22761198324998635


In [ ]:
0.2371832278065893